# 类脑 Phase4 · PYNQ-Z2 点灯示例

> **项目**：neuromorphic-computing · 路径 B 入门  
> **板卡**：PYNQ-Z2 · LD4/LD5 为 RGB 灯

本 Notebook 演示：
1. 加载 `base.bit` overlay
2. 点亮 **LD4 蓝灯**
3. 循环切换 RGB 颜色
4. （可选）模拟脉冲累加 — 有脉冲时闪蓝灯

菜单 **Run → Run All Cells** 即可。

In [ ]:
import os
import time

os.environ.setdefault("XILINX_XRT", "/usr")

from pynq.overlays.base import BaseOverlay

print("Loading base overlay...")
base = BaseOverlay("base.bit")
print("Overlay loaded:", base.is_loaded())

## 1. 蓝灯 LD4

RGB 编码：`1`=蓝，`2`=绿，`4`=红（可相加混色）

In [ ]:
BLUE = 1
LD4 = 4

base.rgbleds[LD4].write(BLUE)
print("LD4 蓝色 ON — 请看板子上的 RGB 灯")
time.sleep(3)
base.rgbleds[LD4].off()
print("LD4 OFF")

## 2. RGB 循环（LD4 + LD5）

In [ ]:
colors = {
    "蓝": 1,
    "绿": 2,
    "红": 4,
    "青": 3,
    "紫": 5,
    "黄": 6,
    "白": 7,
}

for name, code in colors.items():
    base.rgbleds[4].write(code)
    base.rgbleds[5].write(code)
    print(f"颜色: {name} ({code})")
    time.sleep(0.8)

base.rgbleds[4].off()
base.rgbleds[5].off()
print("循环结束")

## 3. 类脑脉冲累加演示（定点 LIF）

与 Phase4 `MnistSNNUnrolled` 相同的 LIF 规则（subtract reset），  
每产生一个脉冲，LD4 蓝灯闪一下 —— **事件驱动**原型。

In [ ]:
FRAC = 16
SCALE = 1 << FRAC
BETA_FP = int(round(0.9 * SCALE))
TH_FP = int(round(1.0 * SCALE))


def lif_step(cur_fp, mem_fp):
    reset = 1 if mem_fp >= TH_FP else 0
    mem_fp = (BETA_FP * mem_fp) // SCALE + cur_fp - reset * TH_FP
    spk = 1 if mem_fp >= TH_FP else 0
    return spk, mem_fp


currents = [0.0, 0.0, 1.2, 0.0, 0.0, 1.1, 1.1, 0.0, 0.0, 1.05]
mem_fp = 0
spikes = []

for t, cur in enumerate(currents):
    cur_fp = int(round(cur * SCALE))
    spk, mem_fp = lif_step(cur_fp, mem_fp)
    spikes.append(spk)
    if spk:
        base.rgbleds[4].write(1)
        print(f"t={t} SPIKE cur={cur}")
        time.sleep(0.5)
        base.rgbleds[4].off()
    else:
        time.sleep(0.15)

print(f"脉冲累加 total = {sum(spikes)}  (类脑输出 spk_sum 的原子操作)")
print("FPGA_LED_DEMO_OK")

---
**下一步**：全连接层权重上 PL（HLS）· 与 Atlas 真 SNN `spk_sum` 抽样对比。  
文档：`docs/phase4_fpga_path_b.md` · `docs/PYNQ_Z2_板卡用法与功能.md`